# NLP Lab 1 - Part 2: Text Classification (Emotion Recognition)
Naive Bayes, Logistic Regression and evaluation metrics, all implemented from scratch with NumPy
and compared against scikit-learn.

In [1]:
import re
import time
import nltk
import numpy as np
from datasets import load_dataset

for pkg in ["gutenberg", "punkt", "punkt_tab", "stopwords"]:
    nltk.download(pkg, quiet=True)

from nltk.corpus import gutenberg, stopwords

RNG_SEED = 42
np.random.seed(RNG_SEED)

c:\Users\yasee\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 0. Load the data
The lab dataset is `avsolatorio/mteb-emotion-avs_triplets`. It is a triplet version of the emotion data,
so we first look at its columns. If it has no `text`/`label` columns we fall back to the original
6-class emotion dataset (`dair-ai/emotion`), which has the same sentences and labels.
All available splits are pooled and re-split **90% train / 10% test** (no separate validation set).

In [2]:
FALLBACK_NAMES = ["sadness", "joy", "love", "anger", "fear", "surprise"] 


def _split_90_10(texts, labels, seed=RNG_SEED):
    idx = np.random.RandomState(seed).permutation(len(texts))
    n = len(idx)
    a = int(0.9 * n)
    take = lambda ids: ([texts[i] for i in ids], [labels[i] for i in ids])
    return take(idx[:a]), take(idx[a:])


def load_data():
    ds = None
    try:
        ds = load_dataset("avsolatorio/mteb-emotion-avs_triplets")
        cols = ds[list(ds.keys())[0]].column_names
        print("Triplets dataset columns:", cols)
        if not ({"text", "label"} <= set(cols)):
            ds = None
    except Exception as e:
        print("Could not use the triplets dataset:", e)
        ds = None
    if ds is None:
        print("Falling back to dair-ai/emotion (same 6 classes, text + label columns).")
        ds = load_dataset("dair-ai/emotion")
    names = None
    first = ds[list(ds.keys())[0]]
    if hasattr(first.features["label"], "names"):
        names = first.features["label"].names
    splits = {k: (list(ds[k]["text"]), list(ds[k]["label"])) for k in ds.keys()}
    # pool every available split, then re-split 90% train / 10% test
    all_x = [t for k in splits for t in splits[k][0]]
    all_y = [l for k in splits for l in splits[k][1]]
    tr, te = _split_90_10(all_x, all_y)
    return tr, te, (names or FALLBACK_NAMES)


(train_x, train_y), (test_x, test_y), CLASS_NAMES = load_data()
train_y, test_y = map(np.array, (train_y, test_y))
C = len(CLASS_NAMES)
print(f"train={len(train_x)}  test={len(test_x)}  classes={CLASS_NAMES}")

Triplets dataset columns: ['text', 'label', 'label_text', 'idx', 'query_idx', 'positive_idx', 'negative_idx']
train=14400  test=1600  classes=['sadness', 'joy', 'love', 'anger', 'fear', 'surprise']


## 1. Preprocessing 
Lowercase -> remove punctuation / digits / special characters -> whitespace tokenization.
Stop-word removal is **off** by default: words like "not", "no", "i", "so" carry emotional signal.

In [3]:
def preprocess(text, stopwords=None):
    text = text.lower()
    text = re.sub(r"[^a-z\s]", " ", text)   # drop punctuation, digits, special characters
    toks = text.split()
    if stopwords:
        toks = [t for t in toks if t not in stopwords]
    return toks


train_tok = [preprocess(t) for t in train_x]
test_tok = [preprocess(t) for t in test_x]
print(train_x[0], "->", train_tok[0])

i do i feel like i just make him mad or upset and he doesn t talk to me -> ['i', 'do', 'i', 'feel', 'like', 'i', 'just', 'make', 'him', 'mad', 'or', 'upset', 'and', 'he', 'doesn', 't', 'talk', 'to', 'me']


## 2.3 Confusion matrix & metrics (NumPy only)
Built first because Parts 2.1 and 2.2 are evaluated with it.
Rows = true class, columns = predicted class.



In [4]:
def confusion_matrix_np(y_true, y_pred, n_classes):
    cm = np.zeros((n_classes, n_classes), dtype=np.int64)
    np.add.at(cm, (y_true, y_pred), 1)
    return cm


def _safe_div(a, b):
    out = np.zeros_like(a, dtype=float)
    np.divide(a, b, out=out, where=(b != 0))
    return out


def prf_from_cm(cm):
    tp = np.diag(cm).astype(float)
    fp = cm.sum(axis=0) - tp
    fn = cm.sum(axis=1) - tp
    precision = _safe_div(tp, tp + fp)
    recall = _safe_div(tp, tp + fn)
    f1 = _safe_div(2 * precision * recall, precision + recall)
    return precision, recall, f1


def evaluate(y_true, y_pred, n_classes, name="model", verbose=True):
    cm = confusion_matrix_np(y_true, y_pred, n_classes)
    p, r, f = prf_from_cm(cm)
    acc = np.trace(cm) / cm.sum()
    res = dict(cm=cm, precision=p, recall=r, f1=f, accuracy=acc,
               macro_p=p.mean(), macro_r=r.mean(), macro_f1=f.mean())
    if verbose:
        print(f"\n=== {name} ===  accuracy={acc:.4f}  macro-P={p.mean():.4f}  "
              f"macro-R={r.mean():.4f}  macro-F1={f.mean():.4f}")
        print(f"{'class':<10}{'prec':>8}{'rec':>8}{'f1':>8}{'support':>9}")
        for k in range(n_classes):
            print(f"{CLASS_NAMES[k]:<10}{p[k]:>8.3f}{r[k]:>8.3f}{f[k]:>8.3f}{cm[k].sum():>9d}")
        print("confusion matrix:\n", cm)
    return res

### Validate the metrics against scikit-learn
(Run on any predictions; we use random ones here so the check is independent of the models.)

In [5]:
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score


def check_against_sklearn(y_true, y_pred, n_classes):
    mine = evaluate(y_true, y_pred, n_classes, verbose=False)
    labels = list(range(n_classes))
    assert np.array_equal(mine["cm"], confusion_matrix(y_true, y_pred, labels=labels))
    for nm, fn in [("precision", precision_score), ("recall", recall_score), ("f1", f1_score)]:
        ref = fn(y_true, y_pred, labels=labels, average=None, zero_division=0)
        assert np.allclose(mine[nm], ref), nm
        key = "macro_" + ("f1" if nm == "f1" else nm[0])
        assert np.isclose(mine[key], fn(y_true, y_pred, labels=labels, average="macro", zero_division=0)), nm
    print("From-scratch confusion matrix / precision / recall / F1 (per class + macro) match scikit-learn.")


_rp = np.random.RandomState(0).randint(0, C, size=len(test_y))
check_against_sklearn(test_y, _rp, C)

From-scratch confusion matrix / precision / recall / F1 (per class + macro) match scikit-learn.


## Naive Bayes

**Prior** (how common is each class):

$$P(c) = \frac{N_c}{N}$$

**Likelihood with Laplace smoothing** (how typical is word $w$ for class $c$):

$$P(w \mid c) = \frac{\text{count}(w,c) + \alpha}{\sum_{w'} \text{count}(w',c) + \alpha \cdot |V|}$$

**Prediction** (pick the class with the highest score, using logs):

$$\hat{c} = \arg\max_{c} \left[ \log P(c) + \sum_{w \in d} \log P(w \mid c) \right]$$

Where:
- $N_c$ = number of training sentences in class $c$
- $N$ = total number of training sentences
- $\text{count}(w,c)$ = times word $w$ appears in class $c$
- $\sum_{w'} \text{count}(w',c)$ = total words in class $c$
- $|V|$ = vocabulary size
- $\alpha$ = smoothing constant (1 for Laplace)
- $d$ = the sentence being classified


In [6]:
class NaiveBayesNP:
    def __init__(self, alpha=1.0):
        self.alpha = alpha

    def fit(self, docs, y, n_classes):
        self.C = n_classes
        self.vocab = {w: i for i, w in enumerate(sorted({w for d in docs for w in d}))}
        V = len(self.vocab)
        counts = np.zeros((n_classes, V))
        for d, c in zip(docs, y):
            ids = np.fromiter((self.vocab[w] for w in d), dtype=np.int64, count=len(d))
            np.add.at(counts[c], ids, 1)                       
        class_counts = np.bincount(y, minlength=n_classes)
        self.log_prior = np.log(class_counts / class_counts.sum())
        self.log_lik = np.log((counts + self.alpha) /(counts.sum(axis=1, keepdims=True) + self.alpha * V))
        return self

    def predict_log_joint(self, docs):
        out = np.empty((len(docs), self.C))
        for i, d in enumerate(docs):
            ids = [self.vocab[w] for w in d if w in self.vocab]
            out[i] = self.log_prior + self.log_lik[:, ids].sum(axis=1)
        return out

    def predict(self, docs):
        return self.predict_log_joint(docs).argmax(axis=1)


t0 = time.time()
nb = NaiveBayesNP(alpha=1.0).fit(train_tok, train_y, C)
nb_pred = nb.predict(test_tok)
print(f"NB from scratch: fit+predict in {time.time()-t0:.2f}s, |V|={len(nb.vocab)}")
res_nb = evaluate(test_y, nb_pred, C, "Naive Bayes (from scratch)")

NB from scratch: fit+predict in 0.12s, |V|=14453

=== Naive Bayes (from scratch) ===  accuracy=0.7450  macro-P=0.8411  macro-R=0.5277  macro-F1=0.5524
class         prec     rec      f1  support
sadness      0.721   0.943   0.818      475
joy          0.715   0.941   0.813      544
love         0.833   0.200   0.323      125
anger        0.885   0.545   0.674      211
fear         0.892   0.523   0.659      174
surprise     1.000   0.014   0.028       71
confusion matrix:
 [[448  23   1   3   0   0]
 [ 26 512   4   2   0   0]
 [ 17  80  25   2   1   0]
 [ 52  43   0 115   1   0]
 [ 47  31   0   5  91   0]
 [ 31  27   0   3   9   1]]


### scikit-learn Naive Bayes (CountVectorizer + MultinomialNB)
`token_pattern` is changed so single-letter words such as "i" are kept, matching our preprocessing.

Specifically, CountVectorizer automates three critical steps to create a "bag-of-words" representation:

Tokenization: It automatically preprocesses the raw text (usually lowercasing) and splits strings into individual words.

Vocabulary Building: It scans the entire training dataset to identify all unique words and assigns a specific column index to each one, replacing your manual self.vocab = {w: i...} logic.

Vectorization: It converts the raw documents into a numerical Document-Term Matrix. For every document, it outputs an array containing the exact count of how many times each vocabulary word appeared. This provides the exact input matrix format that MultinomialNB requires to calculate its probabilities, replacing your custom np.add.at counting loops.

In [7]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB

join = lambda toks_list: [" ".join(t) for t in toks_list]
cv = CountVectorizer(token_pattern=r"(?u)\b\w+\b")
Xtr_cv = cv.fit_transform(join(train_tok))
Xte_cv = cv.transform(join(test_tok))
sk_nb = MultinomialNB(alpha=1.0).fit(Xtr_cv, train_y)
sk_nb_pred = sk_nb.predict(Xte_cv)
res_sk_nb = evaluate(test_y, sk_nb_pred, C, "Naive Bayes (scikit-learn)")
print("\nPredictions identical to scratch NB:", np.mean(sk_nb_pred == nb_pred) * 100, "%")


=== Naive Bayes (scikit-learn) ===  accuracy=0.7450  macro-P=0.8411  macro-R=0.5277  macro-F1=0.5524
class         prec     rec      f1  support
sadness      0.721   0.943   0.818      475
joy          0.715   0.941   0.813      544
love         0.833   0.200   0.323      125
anger        0.885   0.545   0.674      211
fear         0.892   0.523   0.659      174
surprise     1.000   0.014   0.028       71
confusion matrix:
 [[448  23   1   3   0   0]
 [ 26 512   4   2   0   0]
 [ 17  80  25   2   1   0]
 [ 52  43   0 115   1   0]
 [ 47  31   0   5  91   0]
 [ 31  27   0   3   9   1]]

Predictions identical to scratch NB: 100.0 %


## 2.2 Logistic Regression
### Feature representation: binary bi-gram presence
Each sentence gets boundary markers `<s>` ... `</s>` (so one-word sentences still produce a bi-gram),
then every adjacent pair is a feature. The vocabulary is built from **training** sentences only
(building it from the test set would leak information); unseen bi-grams at test time are dropped.
A sentence is stored sparsely as the array of its active feature indices (a 1 in the binary vector).

In [ ]:
def bigrams(toks):
    t = ["<s>"] + list(toks) + ["</s>"]
    return [t[i] + " " + t[i + 1] for i in range(len(t) - 1)]


def build_bigram_vocab(docs):
    return {b: i for i, b in enumerate(sorted({b for d in docs for b in bigrams(d)}))}


def to_sparse_binary(docs, vocab):
    """List of int arrays; each array = indices where the binary feature vector is 1."""
    return [np.unique(np.array([vocab[b] for b in bigrams(d) if b in vocab], dtype=np.int64))
            for d in docs]


bg_vocab = build_bigram_vocab(train_tok)
Vb = len(bg_vocab)
Ftr, Fte = (to_sparse_binary(d, bg_vocab) for d in (train_tok, test_tok))
print(f"bi-gram vocabulary: {Vb} features")

bi-gram vocabulary: 103489 features; avg active per sentence: 20.0


### Discussion: is binary bi-gram presence a good representation?
**Advantages**
- Captures a bit of local word order and context that unigrams miss (e.g. "not happy", "so scared", "i love").
- Binary presence is simple, robust to repeated words (sentences are short), and gives a very sparse vector,
  so training and storage are cheap (we only touch the ~20 active features per sentence).
- Linear models on n-gram features are strong, interpretable baselines.

**Limitations**
- Huge, very sparse feature space (>100k features for ~16k sentences); most bi-grams appear once, so the model
  can overfit and most weights are barely trained.
- Data sparsity: a bi-gram not seen in training gets no feature at all, and similar phrasings
  ("so happy" vs "really glad") share nothing. No notion of synonyms or semantic similarity.
- Only context of one neighbouring word; long-range dependencies and negation scope are lost.
- Binary presence discards counts and word importance (no TF-IDF weighting).
- Dropping stop words / punctuation in preprocessing removes some emotional cues.
- Better alternatives: unigram+bigram combo, TF-IDF, min-frequency cutoff, or dense embeddings.

### Softmax (multinomial) logistic regression from scratch

For a batch of $B$ sentences, each represented by a sparse binary vector $x_i$ (1 = bi-gram present):

**Step 1: Scores** (one score per class)

$$z_i = x_i W + b = \sum_{j \in \text{active}(i)} W[j] + b$$

Only the active features are summed, which is the same as the full product $XW + b$ but much faster.

**Step 2: Probabilities and loss**

$$P_{i,k} = \frac{e^{z_{i,k}}}{\sum_{m} e^{z_{i,m}}}$$

$$L = -\frac{1}{B} \sum_{i=1}^{B} \log P_{i,\,y_i}$$

The maximum score in each row is subtracted before applying the exponential, which avoids overflow and does not change the result.

**Step 3: Gradients**

$$dZ = \frac{P - Y}{B}$$

$$dW[j] = \sum_{i \,:\, j \in \text{active}(i)} dZ_i \qquad\qquad db = \sum_{i=1}^{B} dZ_i$$

**Step 4: Update**

$$W \leftarrow W - \eta \, dW \qquad\qquad b \leftarrow b - \eta \, db$$

The dataset is **shuffled at the start of every epoch**.

Where:
- $W$ = weight matrix of shape (number of bi-grams, number of classes)
- $b$ = bias vector, one value per class
- $P$ = predicted class probabilities
- $Y$ = one-hot true labels, so $P - Y$ is the error for each class
- $y_i$ = true class of sentence $i$
- $B$ = batch size
- $\eta$ = learning rate

In [9]:
def softmax(z):
    z = z - z.max(axis=1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=1, keepdims=True)


class LogisticRegressionNP:
    def __init__(self, lr=0.5, epochs=15, batch_size=64, seed=RNG_SEED):
        self.lr, self.epochs, self.bs, self.seed = lr, epochs, batch_size, seed

    @staticmethod
    def _flatten(batch):
        lens = np.fromiter((len(a) for a in batch), dtype=np.int64, count=len(batch))
        cols = np.concatenate(batch) if lens.sum() else np.empty(0, dtype=np.int64)
        rows = np.repeat(np.arange(len(batch)), lens)
        return rows, cols

    def _scores(self, batch):
        rows, cols = self._flatten(batch)
        z = np.tile(self.b, (len(batch), 1))
        np.add.at(z, rows, self.W[cols])
        return z, rows, cols

    def fit(self, feats, y, n_features, n_classes):
        rng = np.random.RandomState(self.seed)
        self.W = np.zeros((n_features, n_classes))       # zero init is fine for (softmax) LR: convex loss
        self.b = np.zeros(n_classes)
        N = len(feats)
        self.history = []
        for ep in range(self.epochs):
            order = rng.permutation(N)                   
            tot_loss = 0.0
            for s in range(0, N, self.bs):
                ids = order[s:s + self.bs]
                batch = [feats[i] for i in ids]
                yb = y[ids]
                z, rows, cols = self._scores(batch)       # 1. scores
                P = softmax(z)                            # 2. probabilities + loss
                tot_loss += -np.log(P[np.arange(len(ids)), yb] + 1e-12).sum()
                dZ = P.copy()                             # 3. gradients
                dZ[np.arange(len(ids)), yb] -= 1
                dZ /= len(ids)
                np.add.at(self.W, cols, -self.lr * dZ[rows])   # 4. updates
                self.b -= self.lr * dZ.sum(axis=0)
            rec = {"epoch": ep + 1, "train_loss": tot_loss / N}
            self.history.append(rec)
            print("  ", {k: (round(v, 4) if isinstance(v, float) else v) for k, v in rec.items()})
        return self

    def predict_proba(self, feats):
        return softmax(self._scores(feats)[0])

    def predict(self, feats):
        return self.predict_proba(feats).argmax(axis=1)


t0 = time.time()
lr_np = LogisticRegressionNP(lr=0.5, epochs=15, batch_size=64).fit(Ftr, train_y, Vb, C)
print(f"trained in {time.time()-t0:.1f}s")
lr_pred = lr_np.predict(Fte)
res_lr = evaluate(test_y, lr_pred, C, "Logistic Regression (from scratch)")

   {'epoch': 1, 'train_loss': np.float64(1.5363)}
   {'epoch': 2, 'train_loss': np.float64(1.3678)}
   {'epoch': 3, 'train_loss': np.float64(1.2389)}
   {'epoch': 4, 'train_loss': np.float64(1.1304)}
   {'epoch': 5, 'train_loss': np.float64(1.0383)}
   {'epoch': 6, 'train_loss': np.float64(0.9587)}
   {'epoch': 7, 'train_loss': np.float64(0.8893)}
   {'epoch': 8, 'train_loss': np.float64(0.828)}
   {'epoch': 9, 'train_loss': np.float64(0.7745)}
   {'epoch': 10, 'train_loss': np.float64(0.7262)}
   {'epoch': 11, 'train_loss': np.float64(0.6838)}
   {'epoch': 12, 'train_loss': np.float64(0.6453)}
   {'epoch': 13, 'train_loss': np.float64(0.6109)}
   {'epoch': 14, 'train_loss': np.float64(0.5793)}
   {'epoch': 15, 'train_loss': np.float64(0.5508)}
trained in 1.0s

=== Logistic Regression (from scratch) ===  accuracy=0.6319  macro-P=0.6642  macro-R=0.4356  macro-F1=0.4614
class         prec     rec      f1  support
sadness      0.640   0.777   0.702      475
joy          0.587   0.869   0.

### Comparison with scikit-learn `LogisticRegression` and `SGDClassifier`
Same bi-gram features (same vocabulary, binary), so only the optimiser/regulariser differ.
Note sklearn's `LogisticRegression` uses L2 regularisation (C=1) and an L-BFGS solver by default,
while `SGDClassifier(loss="log_loss")` uses SGD with a (small) L2 penalty and a built-in learning-rate schedule.

In [10]:
from sklearn.linear_model import LogisticRegression, SGDClassifier
from scipy.sparse import csr_matrix


def to_csr(feats, n_features):
    indptr = np.concatenate([[0], np.cumsum([len(f) for f in feats])])
    indices = np.concatenate(feats)
    return csr_matrix((np.ones(len(indices)), indices, indptr), shape=(len(feats), n_features))


Xtr, Xte = to_csr(Ftr, Vb), to_csr(Fte, Vb)

t0 = time.time()
sk_lr = LogisticRegression(max_iter=1000).fit(Xtr, train_y)
t_lr = time.time() - t0
t0 = time.time()
sk_sgd = SGDClassifier(loss="log_loss", max_iter=50, tol=1e-4, random_state=RNG_SEED).fit(Xtr, train_y)
t_sgd = time.time() - t0
res_sk_lr = evaluate(test_y, sk_lr.predict(Xte), C, "sklearn LogisticRegression")
res_sk_sgd = evaluate(test_y, sk_sgd.predict(Xte), C, "sklearn SGDClassifier(log_loss)")
print(f"\nfit time: sklearn LR {t_lr:.1f}s | sklearn SGD {t_sgd:.1f}s")


=== sklearn LogisticRegression ===  accuracy=0.6825  macro-P=0.7096  macro-R=0.5201  macro-F1=0.5597
class         prec     rec      f1  support
sadness      0.670   0.802   0.730      475
joy          0.653   0.857   0.741      544
love         0.605   0.208   0.310      125
anger        0.812   0.531   0.642      211
fear         0.797   0.540   0.644      174
surprise     0.722   0.183   0.292       71
confusion matrix:
 [[381  74   2  14   4   0]
 [ 58 466  12   4   3   1]
 [ 34  62  26   0   3   0]
 [ 44  52   0 112   2   1]
 [ 36  33   2   6  94   3]
 [ 16  27   1   2  12  13]]

=== sklearn SGDClassifier(log_loss) ===  accuracy=0.6806  macro-P=0.7239  macro-R=0.5224  macro-F1=0.5686
class         prec     rec      f1  support
sadness      0.717   0.758   0.737      475
joy          0.614   0.895   0.728      544
love         0.622   0.224   0.329      125
anger        0.847   0.498   0.627      211
fear         0.816   0.534   0.646      174
surprise     0.727   0.225   0.344   

## Final comparison table (test set)

In [11]:
rows = [("Naive Bayes (scratch)", res_nb), ("Naive Bayes (sklearn)", res_sk_nb),
        ("LogReg (scratch)", res_lr), ("LogReg (sklearn)", res_sk_lr), ("SGDClassifier (sklearn)", res_sk_sgd)]
print(f"{'model':<26}{'acc':>8}{'macro-P':>9}{'macro-R':>9}{'macro-F1':>10}")
for name, r in rows:
    print(f"{name:<26}{r['accuracy']:>8.4f}{r['macro_p']:>9.4f}{r['macro_r']:>9.4f}{r['macro_f1']:>10.4f}")

agree = lambda a, b: np.mean(a == b) * 100
print(f"\nprediction agreement: scratch LR vs sklearn LR = {agree(lr_pred, sk_lr.predict(Xte)):.1f}% | "
      f"scratch LR vs SGDClassifier = {agree(lr_pred, sk_sgd.predict(Xte)):.1f}%")

model                          acc  macro-P  macro-R  macro-F1
Naive Bayes (scratch)       0.7450   0.8411   0.5277    0.5524
Naive Bayes (sklearn)       0.7450   0.8411   0.5277    0.5524
LogReg (scratch)            0.6319   0.6642   0.4356    0.4614
LogReg (sklearn)            0.6825   0.7096   0.5201    0.5597
SGDClassifier (sklearn)     0.6806   0.7239   0.5224    0.5686

prediction agreement: scratch LR vs sklearn LR = 88.6% | scratch LR vs SGDClassifier = 88.8%


## Discussion (fill in the numbers from your run)
**Which approach performs better, and under what conditions?**
- Compare the table above. Typically scikit-learn's `LogisticRegression` (L-BFGS, run to convergence, with L2
  regularisation) is the best or tied for best: it solves the convex problem to optimality and the L2 term
  limits overfitting on the very sparse, high-dimensional bi-gram features.
- Our SGD version and `SGDClassifier` land close to it when the learning rate / number of epochs is reasonable;
  with too large a learning rate the loss oscillates, with too few epochs it underfits. Without regularisation,
  our model can overfit (training loss keeps falling while test accuracy stalls). With the 90/10 split there is
  no validation set, so hyper-parameters (learning rate, epochs) are fixed in advance rather than tuned on the test set.
- Naive Bayes is far faster (one pass of counting, no iterations) and works well with little data, but its
  independence assumption makes its probabilities poorly calibrated; LR usually wins on macro-F1 when there
  is enough data, particularly on the smaller classes (love, surprise).
- Class imbalance matters: accuracy is dominated by joy/sadness, so macro-F1 is the fairer metric.

**Which implementation is most similar to ours?**
- `SGDClassifier(loss="log_loss")`: both minimise the cross-entropy / log-loss with stochastic (mini-batch)
  gradient descent updates and shuffle each epoch. Differences: SGDClassifier trains one-vs-rest binary
  classifiers (ours is a single multinomial softmax), adds a small L2 penalty by default, and uses an
  automatic learning-rate schedule (ours is a constant learning rate).
- `LogisticRegression` optimises (nearly) the same multinomial objective, but with a deterministic
  second-order-ish solver (L-BFGS) and L2 regularisation rather than SGD.